# Sensitivity Analysis

## Objective

This notebook evaluates how the baseline credit risk score responds to **controlled changes in borrower characteristics**.

The goal is to test whether the scoring framework behaves in a **directionally sensible** and **internally consistent** way when selected risk factors are adjusted.

In this notebook, I do not re-estimate a model.  
Instead, I keep the same rule-based scoring framework from the baseline notebook and apply small, interpretable shocks to several borrower-level variables.

---

## Why Sensitivity Analysis Matters

Sensitivity analysis is useful because it helps answer the following question:

> If one or more borrower characteristics worsen slightly, does the implied probability of default move in the expected direction?

This is important in credit risk and model validation because a reasonable scoring system should be:

- interpretable
- stable
- directionally consistent

If a model produces counterintuitive responses to small changes in key inputs, that may indicate a problem in the framework.

---

## Sensitivity Design

This notebook applies a modest deterioration to selected borrower characteristics:

- **Credit amount** increases by 10%
- **Duration** increases by 20%
- **Saving accounts** deteriorate by one category where possible

These changes are designed to represent mild but plausible worsening in borrower financial conditions.

---

## Expected Outcome

Based on credit risk intuition, these changes should increase the borrower risk score and raise the pseudo probability of default.

In particular:

- larger loan size should increase repayment burden
- longer duration should increase uncertainty
- weaker savings position should reduce liquidity strength

Therefore, the average pseudo PD under the sensitivity scenario is expected to be higher than under the baseline scenario.

---

## Output

This notebook compares:

- baseline pseudo PD
- sensitivity scenario pseudo PD
- change in pseudo PD at the borrower level
- average change in risk across the sample

The results provide a simple validation check on whether the scoring framework reacts to borrower deterioration in a reasonable way.

In [1]:
import pandas as pd
import numpy as np

# ======================
# 1. Read data
# ======================
df = pd.read_csv("/Users/quentingao/Desktop/german_credit_data.csv")

if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

print("Columns:")
print(df.columns.tolist())
print("\nShape:", df.shape)

# ======================
# 2. Basic cleaning
# ======================
df = df.copy()

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()

df = df.replace({"nan": np.nan, "NA": np.nan})

# ======================
# 3. Baseline scoring maps
# ======================
saving_map = {
    "little": 0,
    "moderate": 1,
    "quite rich": 2,
    "rich": 3
}

checking_map = {
    "little": 0,
    "moderate": 1,
    "rich": 2
}

housing_map = {
    "rent": 0,
    "free": 1,
    "own": 2
}

# ======================
# 4. Build baseline dataset
# ======================
base = df.copy()

base["saving_score"] = base["Saving accounts"].map(saving_map)
base["saving_score"] = base["saving_score"].fillna(0.5)

base["checking_score"] = base["Checking account"].map(checking_map)
base["checking_score"] = base["checking_score"].fillna(0.5)

base["housing_score"] = base["Housing"].map(housing_map)
base["job_score"] = base["Job"]

# Standardize continuous variables using baseline moments
continuous_vars = ["Age", "Credit amount", "Duration"]

baseline_means = {}
baseline_stds = {}

for col in continuous_vars:
    baseline_means[col] = base[col].mean()
    baseline_stds[col] = base[col].std()
    base[f"{col}_z"] = (base[col] - baseline_means[col]) / baseline_stds[col]

# Baseline risk score
base["risk_score"] = (
    0.60 * base["Credit amount_z"]
    + 0.75 * base["Duration_z"]
    - 0.25 * base["Age_z"]
    - 0.80 * base["saving_score"]
    - 0.60 * base["checking_score"]
    - 0.30 * base["housing_score"]
    - 0.20 * base["job_score"]
)

base["pseudo_pd"] = 1 / (1 + np.exp(-base["risk_score"]))

# ======================
# 5. Build sensitivity scenario
# ======================
sens = df.copy()

# Apply mild deterioration shocks
sens["Credit amount"] = sens["Credit amount"] * 1.10
sens["Duration"] = sens["Duration"] * 1.20

# Saving accounts deteriorate by one level where possible
sens["Saving accounts"] = sens["Saving accounts"].replace({
    "moderate": "little",
    "quite rich": "moderate",
    "rich": "quite rich"
})

# Rebuild scores under sensitivity scenario
sens["saving_score"] = sens["Saving accounts"].map(saving_map)
sens["saving_score"] = sens["saving_score"].fillna(0.5)

sens["checking_score"] = sens["Checking account"].map(checking_map)
sens["checking_score"] = sens["checking_score"].fillna(0.5)

sens["housing_score"] = sens["Housing"].map(housing_map)
sens["job_score"] = sens["Job"]

# IMPORTANT:
# Use baseline means/stds so the comparison is consistent
for col in continuous_vars:
    sens[f"{col}_z"] = (sens[col] - baseline_means[col]) / baseline_stds[col]

# Sensitivity risk score
sens["risk_score"] = (
    0.60 * sens["Credit amount_z"]
    + 0.75 * sens["Duration_z"]
    - 0.25 * sens["Age_z"]
    - 0.80 * sens["saving_score"]
    - 0.60 * sens["checking_score"]
    - 0.30 * sens["housing_score"]
    - 0.20 * sens["job_score"]
)

sens["pseudo_pd"] = 1 / (1 + np.exp(-sens["risk_score"]))

# ======================
# 6. Compare baseline vs sensitivity
# ======================
comparison = pd.DataFrame({
    "baseline_pd": base["pseudo_pd"],
    "sensitivity_pd": sens["pseudo_pd"]
})

comparison["delta_pd"] = comparison["sensitivity_pd"] - comparison["baseline_pd"]

print("Baseline average pseudo PD:", round(comparison["baseline_pd"].mean(), 6))
print("Sensitivity average pseudo PD:", round(comparison["sensitivity_pd"].mean(), 6))
print("Average change in pseudo PD:", round(comparison["delta_pd"].mean(), 6))

print("\nDistribution of borrower-level PD changes:")
print(comparison["delta_pd"].describe())

print("\nShare of borrowers with higher PD after sensitivity shock:")
print(round((comparison["delta_pd"] > 0).mean(), 4))

# ======================
# 7. Risk bucket comparison
# ======================
base["risk_bucket"] = pd.cut(
    base["pseudo_pd"],
    bins=[0, 0.20, 0.40, 0.60, 1.00],
    labels=["Low", "Moderate", "Elevated", "High"],
    include_lowest=True
)

sens["risk_bucket"] = pd.cut(
    sens["pseudo_pd"],
    bins=[0, 0.20, 0.40, 0.60, 1.00],
    labels=["Low", "Moderate", "Elevated", "High"],
    include_lowest=True
)

print("\nBaseline risk bucket distribution:")
print(base["risk_bucket"].value_counts().sort_index())

print("\nSensitivity risk bucket distribution:")
print(sens["risk_bucket"].value_counts().sort_index())

# ======================
# 8. Top borrowers with largest PD increase
# ======================
result = df.copy()
result["baseline_pd"] = base["pseudo_pd"]
result["sensitivity_pd"] = sens["pseudo_pd"]
result["delta_pd"] = comparison["delta_pd"]

print("\nTop 10 borrowers with largest PD increase:")
print(
    result[
        [
            "Age", "Sex", "Job", "Housing", "Saving accounts",
            "Checking account", "Credit amount", "Duration", "Purpose",
            "baseline_pd", "sensitivity_pd", "delta_pd"
        ]
    ]
    .sort_values("delta_pd", ascending=False)
    .head(10)
)

Columns:
['Age', 'Sex', 'Job', 'Housing', 'Saving accounts', 'Checking account', 'Credit amount', 'Duration', 'Purpose']

Shape: (1000, 9)
Baseline average pseudo PD: 0.237174
Sensitivity average pseudo PD: 0.304757
Average change in pseudo PD: 0.067583

Distribution of borrower-level PD changes:
count    1000.000000
mean        0.067583
std         0.062820
min         0.001914
25%         0.020300
50%         0.048649
75%         0.096885
max         0.377919
Name: delta_pd, dtype: float64

Share of borrowers with higher PD after sensitivity shock:
1.0

Baseline risk bucket distribution:
risk_bucket
Low         600
Moderate    205
Elevated     96
High         99
Name: count, dtype: int64

Sensitivity risk bucket distribution:
risk_bucket
Low         484
Moderate    241
Elevated    115
High        160
Name: count, dtype: int64

Top 10 borrowers with largest PD increase:
     Age     Sex  Job Housing Saving accounts Checking account  Credit amount  \
226   27    male    2     own      

# Sensitivity Analysis: Results Summary

## Main Finding

The sensitivity scenario produces a higher average pseudo probability of default than the baseline scenario.

This indicates that the rule-based scoring framework responds in the expected direction when borrower conditions deteriorate modestly.

---

## Interpretation

The sensitivity shocks were designed to worsen borrower risk through:

- larger credit exposure
- longer repayment horizon
- weaker savings position

All three changes should increase credit risk, so an upward shift in pseudo PD is economically intuitive.

---

## Borrower-Level Impact

The borrower-level comparison shows that most borrowers experience an increase in pseudo PD under the sensitivity scenario.

This suggests that the scoring framework is responsive to mild deterioration in financial conditions and behaves consistently across the portfolio.

---

## Risk Segmentation

Comparing risk buckets before and after the sensitivity shock helps show whether borrowers migrate from lower-risk categories into more elevated risk groups.

This type of migration analysis is useful for understanding how a credit portfolio may respond even to relatively small adverse changes.

---

## Conclusion

Overall, the sensitivity analysis supports the internal consistency of the scoring framework.

Small, interpretable deteriorations in borrower characteristics lead to higher implied default risk, which is consistent with standard credit risk intuition.

This baseline sensitivity exercise provides a useful bridge to the next step: **stress testing under more severe scenarios**.